# Notebook 07 — Dashboard Data Prep

**Goal:** Produce two dashboard-ready CSVs for upload to Google Sheets, which Looker Studio will connect to. This notebook adds all derived columns that Looker Studio would struggle to compute itself (season labels, quarter start dates, performance bands, percentage display values).

**Inputs:**
- `data/processed/national_monthly.csv` — 85 rows
- `data/processed/quarterly_by_provider.csv` — 5,062 rows

**Outputs:**
- `data/processed/dashboard_national.csv` — national monthly, enriched
- `data/processed/dashboard_trust.csv` — trust-level quarterly, enriched

**Then upload both CSVs to Google Sheets and connect Looker Studio — see the design spec in `outputs/reports/dashboard_design.md`.**

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed")

national  = pd.read_csv(DATA_DIR / "national_monthly.csv",      parse_dates=["period"])
quarterly = pd.read_csv(DATA_DIR / "quarterly_by_provider.csv")

print(f"national:  {national.shape}")
print(f"quarterly: {quarterly.shape}")

national:  (85, 22)
quarterly: (5062, 30)


## National monthly — add derived columns

In [2]:
# Financial year
def fy(dt):
    if dt.month >= 4:
        return f"{dt.year}-{str(dt.year + 1)[2:]}"
    return f"{dt.year - 1}-{str(dt.year)[2:]}"

national["financial_year"] = national["period"].apply(fy)

# Calendar fields
national["month"]      = national["period"].dt.month
national["month_name"] = national["period"].dt.strftime("%b")
national["year"]       = national["period"].dt.year

# FY month position: Apr=1 … Mar=12  (for seasonal profiles Apr-Mar)
national["fy_month_pos"] = national["month"].apply(lambda m: m - 3 if m >= 4 else m + 9)

# Season label (calendar)
def season(m):
    if m in (12, 1, 2):  return "Winter"
    if m in (3, 4, 5):   return "Spring"
    if m in (6, 7, 8):   return "Summer"
    return "Autumn"

national["season"] = national["month"].apply(season)

# Era label
def era(dt):
    if dt < pd.Timestamp("2020-03-01"):  return "Pre-COVID"
    if dt <= pd.Timestamp("2021-09-01"): return "COVID"
    return "Post-COVID"

national["era"] = national["period"].apply(era)

# Convert decimal pct columns → percentage (0.753 → 75.3)
pct_cols = [c for c in national.columns if c.startswith("pct_")]
for col in pct_cols:
    national[f"{col}_pct"] = (national[col] * 100).round(1)

# At-target flag (national Type 1 ≥ 95%)
national["at_target"] = (national["pct_4hr_type1"] >= 0.95).astype(int)

# Pre-pandemic index (April 2019 = 100) for attendance and performance
base_row = national[national["period"] == "2019-04-01"].iloc[0]
national["attendance_index"]   = (national["type1_attendances"] / base_row["type1_attendances"] * 100).round(1)
national["performance_index"]  = (national["pct_4hr_type1"]    / base_row["pct_4hr_type1"]    * 100).round(1)

# Period as YYYY-MM-DD string for Looker Studio date parsing
national["period"] = national["period"].dt.strftime("%Y-%m-%d")

print(f"National enriched: {national.shape}")
print(national[["period","financial_year","month_name","season","era",
                "pct_4hr_type1_pct","at_target","attendance_index"]].head(6))

National enriched: (85, 36)
       period financial_year month_name  season        era  pct_4hr_type1_pct  \
0  2019-04-01        2019-20        Apr  Spring  Pre-COVID               77.1   
1  2019-05-01        2019-20        May  Spring  Pre-COVID               79.1   
2  2019-06-01        2019-20        Jun  Summer  Pre-COVID               78.8   
3  2019-07-01        2019-20        Jul  Summer  Pre-COVID               78.9   
4  2019-08-01        2019-20        Aug  Summer  Pre-COVID               78.3   
5  2019-09-01        2019-20        Sep  Autumn  Pre-COVID               77.0   

   at_target  attendance_index  
0          0             100.0  
1          0             102.9  
2          0             100.2  
3          0             106.4  
4          0              99.5  
5          0             100.9  


## Trust quarterly — add derived columns

In [3]:
# Quarter start date — needed for time-series charts in Looker Studio
# Q1 = Apr, Q2 = Jul, Q3 = Oct, Q4 = Jan of next calendar year
Q_START_MONTH = {1: 4, 2: 7, 3: 10, 4: 1}

def quarter_start_date(row):
    fy_str = row["financial_year"]   # e.g. "2024-25"
    fy_year = int(fy_str[:4])        # e.g. 2024
    q = int(row["quarter"])
    month = Q_START_MONTH[q]
    year  = fy_year if month >= 4 else fy_year + 1
    return f"{year:04d}-{month:02d}-01"

quarterly["quarter_start_date"] = quarterly.apply(quarter_start_date, axis=1)
quarterly["fy_quarter"]         = quarterly["financial_year"] + " Q" + quarterly["quarter"].astype(str)

# Region short labels
REGION_SHORT = {
    "North East And Yorkshire":           "NE & Yorkshire",
    "North West":                         "North West",
    "Midlands":                           "Midlands",
    "East Of England":                    "East of England",
    "London":                             "London",
    "South East":                         "South East",
    "South West":                         "South West",
}
quarterly["region_short"] = quarterly["region"].map(REGION_SHORT).fillna(quarterly["region"])

# Performance band (Type 1)
def perf_band(v):
    if pd.isna(v):    return None
    if v >= 0.95:     return "1. >= 95% (target)"
    if v >= 0.85:     return "2. 85-94%"
    if v >= 0.75:     return "3. 75-84%"
    return                   "4. < 75% (severe)"

quarterly["performance_band"] = quarterly["pct_4hr_type1"].apply(perf_band)

# At-target flag
quarterly["at_target"] = (quarterly["pct_4hr_type1"] >= 0.95).astype("Int8")

# Convert decimal pct columns → percentage
q_pct_cols = [c for c in quarterly.columns if c.startswith("pct_")]
for col in q_pct_cols:
    quarterly[f"{col}_pct"] = (quarterly[col] * 100).round(1)

# Local trust flag
quarterly["is_local_trust"] = (quarterly["code"] == "RGN").astype(int)

print(f"Trust enriched: {quarterly.shape}")
print(quarterly[["code","name","financial_year","quarter","quarter_start_date",
                  "fy_quarter","region_short","performance_band","pct_4hr_type1_pct"]].head(4))

Trust enriched: (5062, 40)
    code                                               name financial_year  \
0    RDD  Basildon And Thurrock University Hospitals NHS...        2019-20   
1    RC1                         Bedford Hospital NHS Trust        2019-20   
2    RGT  Cambridge University Hospitals NHS Foundation ...        2019-20   
3  NQ108                                   Clacton Hospital        2019-20   

   quarter quarter_start_date  fy_quarter                  region_short  \
0        1         2019-04-01  2019-20 Q1  NHS England East Of England    
1        1         2019-04-01  2019-20 Q1  NHS England East Of England    
2        1         2019-04-01  2019-20 Q1  NHS England East Of England    
3        1         2019-04-01  2019-20 Q1  NHS England East Of England    

  performance_band  pct_4hr_type1_pct  
0        2. 85-94%               94.0  
1        3. 75-84%               80.3  
2             None                NaN  
3             None                NaN  


## Export dashboard CSVs

In [4]:
national.to_csv(DATA_DIR / "dashboard_national.csv", index=False)
quarterly.to_csv(DATA_DIR / "dashboard_trust.csv",   index=False)

print("Exported:")
print(f"  dashboard_national.csv  — {len(national):,} rows × {len(national.columns)} cols")
print(f"  dashboard_trust.csv     — {len(quarterly):,} rows × {len(quarterly.columns)} cols")
print()
print("Columns in dashboard_national.csv:")
print([c for c in national.columns])
print()
print("Columns in dashboard_trust.csv:")
print([c for c in quarterly.columns])

Exported:
  dashboard_national.csv  — 85 rows × 36 cols
  dashboard_trust.csv     — 5,062 rows × 40 cols

Columns in dashboard_national.csv:
['period', 'type1_attendances', 'type2_attendances', 'type3_attendances', 'total_attendances', 'emerg_admissions_type1', 'emerg_admissions_type2', 'emerg_admissions_type3', 'total_emerg_admissions_ae', 'other_emerg_admissions', 'total_emerg_admissions', 'dtoa_over_4hr', 'dtoa_over_12hr', 'total_seen_4hr', 'type1_seen_4hr', 'type2_seen_4hr', 'type3_seen_4hr', 'total_breach_4hr', 'pct_4hr_all', 'pct_4hr_type1', 'pct_4hr_type2', 'pct_4hr_type3', 'financial_year', 'month', 'month_name', 'year', 'fy_month_pos', 'season', 'era', 'pct_4hr_all_pct', 'pct_4hr_type1_pct', 'pct_4hr_type2_pct', 'pct_4hr_type3_pct', 'at_target', 'attendance_index', 'performance_index']

Columns in dashboard_trust.csv:
['code', 'region', 'name', 'type1_attendances', 'type2_attendances', 'type3_attendances', 'total_attendances', 'type1_seen_4hr', 'type2_seen_4hr', 'type3_seen_4h

## Upload to Google Sheets — step-by-step

Looker Studio cannot read local files directly. The data must go via Google Sheets.

1. Go to [sheets.google.com](https://sheets.google.com) and create two new sheets:
   - `NHS AE — National Monthly`
   - `NHS AE — Trust Quarterly`

2. In each sheet: **File → Import → Upload** → select the CSV → *Replace current sheet* → *No conversion*
   - Upload `dashboard_national.csv` into the first sheet
   - Upload `dashboard_trust.csv` into the second sheet

3. In the national sheet, format the `period` column as a date: select column A → Format → Number → Date

4. In the trust sheet, format `quarter_start_date` as a date the same way

5. Go to [lookerstudio.google.com](https://lookerstudio.google.com) → **Create → Report**

6. Add data source → **Google Sheets** → select your sheet → tick *Use first row as headers*

7. Confirm field types:
   - `period` / `quarter_start_date` → Date
   - All `_pct` columns → Number (not Percentage — they're already converted to 0-100 scale)
   - `financial_year`, `region`, `name`, `code` → Text

See `outputs/reports/dashboard_design.md` for the full dashboard layout spec.